# LeetCode 4027 — Elevator Requests III bằng Python

- Link: https://leetcode.com/problems/elevator-requests-iii/
- Độ khó: **Hard**
- Chủ đề: Quy hoạch động bitmask (Held–Karp / người bán hàng), Sắp xếp, Mô phỏng thời gian chờ

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def elevatorRequests(self, n: int, start: int, requests: list[list[int]]) -> int:
```

## 1. Giới thiệu & mục tiêu học

Thang máy bắt đầu ở tầng `start` lúc thời điểm 0. Mỗi yêu cầu `[arrival, floor]` chỉ phục vụ được khi thang ở tầng `floor` vào lúc **không sớm hơn** `arrival`. Mỗi giây thang đi lên 1, xuống 1 hoặc đứng yên. Tìm thời điểm nhỏ nhất mà **mọi** yêu cầu được phục vụ. Ràng buộc quan trọng: chỉ có **tối đa 16 yêu cầu** (nhưng `n` và `arrival` tới `10^9`).

Với tối đa 16 yêu cầu, bài chính là **bài toán người bán hàng (TSP)** có "cửa sổ thời gian": chọn **thứ tự** ghé các tầng. Cách giải chuẩn là **DP bitmask** (Held–Karp): `m² · 2^m` thay vì `m!`.

Sau notebook này bạn sẽ:

1. Mô hình hoá một thứ tự phục vụ: `t = max(t + |tầng_trước − tầng_sau|, arrival)`.
2. Học **bitmask** từ đầu: biểu diễn tập con bằng một số nguyên, các thao tác `|`, `&`, `^`, `<<`.
3. Hiểu vì sao với mỗi trạng thái chỉ cần giữ **thời điểm nhỏ nhất** (được phép chờ).
4. Cài DP bitmask 3 kiểu: bottom-up, top-down có nhớ, và bản tối ưu chỉ duyệt các bit bật.
5. Đối chiếu với BFS theo từng giây (nguyên văn đề), hoán vị và DP đoạn `O(m²)` (dành cho bản IV) bằng test ngẫu nhiên có seed.

## 2. Đề bài & ví dụ

**Phát biểu lại.**

- Tầng đánh số `0..n-1`. Thang ở tầng `start` lúc `0`.
- Mỗi giây: lên 1 tầng, xuống 1 tầng, hoặc đứng yên.
- `requests[i] = [arrival_i, floor_i]`: yêu cầu tầng `floor_i` xuất hiện lúc `arrival_i`. Nó được phục vụ **ngay** khi thang ở tầng đó tại **bất kỳ thời điểm nào ≥ `arrival_i`**.
- Trả về thời điểm **nhỏ nhất** để phục vụ xong **mọi** yêu cầu.

**Ví dụ 1:** `n = 9, start = 0, requests = [[0,8],[6,5]]` → `9`.
**Ví dụ 2:** `n = 8, start = 5, requests = [[1,7],[7,3]]` → `7`.
**Ví dụ 3:** `n = 7, start = 3, requests = [[0,5],[0,1],[6,3]]` → `8`.

**Ràng buộc**

- `1 <= n <= 10^9`
- `1 <= requests.length <= 16`
- `0 <= arrival_i <= 10^9`
- `0 <= start, floor_i <= n - 1`

(Các tầng không bắt buộc khác nhau, nên có thể có hai yêu cầu cùng tầng.)

In [1]:
EXAMPLES = [
    (9, 0, [[0, 8], [6, 5]], 9),
    (8, 5, [[1, 7], [7, 3]], 7),
    (7, 3, [[0, 5], [0, 1], [6, 3]], 8),
]
for n, start, reqs, e in EXAMPLES:
    print(f"n={n}, start={start}, requests={reqs} -> kỳ vọng {e}")

n=9, start=0, requests=[[0, 8], [6, 5]] -> kỳ vọng 9
n=8, start=5, requests=[[1, 7], [7, 3]] -> kỳ vọng 7
n=7, start=3, requests=[[0, 5], [0, 1], [6, 3]] -> kỳ vọng 8


## 3. Kiến thức nền

### 3.1. Khoảng cách và "đến sớm thì chờ"

Đi từ tầng `p` tới tầng `f` mất `abs(p - f)` giây. Nếu rời `p` ở thời điểm `t` thì đến `f` lúc `t + abs(p - f)`. Yêu cầu chỉ phục vụ từ `arrival` trở đi, nên thời điểm phục vụ là
`max(t + abs(p - f), arrival)` — đến sớm thì **đứng chờ**.

In [2]:
def step(t, pos, floor, arrival):
    return max(t + abs(pos - floor), arrival)

print(step(0, 0, 5, 6))    # đến lúc 5, chờ tới 6 -> 6
print(step(6, 5, 8, 0))    # 6 + 3 = 9
print(step(0, 3, 3, 0))    # đứng sẵn đúng tầng, arrival 0 -> 0

6
9
0


### 3.2. Một thứ tự phục vụ cho trước

Cho `order` là danh sách các `(arrival, floor)` theo thứ tự phục vụ, ta tính thời điểm cuối bằng một vòng lặp áp dụng `step`. Bài toán là **chọn thứ tự tốt nhất**.

In [3]:
def finish_time(start, order):
    t, pos = 0, start
    for arrival, floor in order:
        t = step(t, pos, floor, arrival)
        pos = floor
    return t

print(finish_time(3, [(0, 5), (0, 1), (6, 3)]))   # ví dụ 3, thứ tự 5 -> 1 -> 3: 8
print(finish_time(3, [(0, 1), (0, 5), (6, 3)]))   # thứ tự 1 -> 5 -> 3
print(finish_time(3, [(6, 3), (0, 5), (0, 1)]))   # phục vụ tầng 3 trước: phải chờ tới 6 rồi mới đi tiếp

8
8
12


### 3.3. Bitmask: dùng một số nguyên để biểu diễn tập con

Với `m` phần tử (đánh số `0..m-1`), một **tập con** được lưu bằng số nguyên `mask` mà **bit thứ `j`** bật ⇔ phần tử `j` thuộc tập.

| Thao tác | Ý nghĩa | Ví dụ |
|---|---|---|
| `1 << j` | tập chỉ chứa phần tử `j` | `1 << 3 = 8` |
| `mask \| (1 << j)` | thêm `j` vào tập | |
| `mask & (1 << j)` | khác 0 nếu `j` thuộc tập | |
| `mask ^ (1 << j)` | bỏ `j` ra khỏi tập (khi `j` đang thuộc) | |
| `(1 << m) - 1` | tập đầy đủ `{0..m-1}` | `m=3 → 7` |

In [4]:
m = 4
mask = 0
for j in (0, 2):
    mask |= 1 << j
print(bin(mask), mask)                                   # 0b101 = 5
print([bool(mask & (1 << j)) for j in range(m)])         # [True, False, True, False]
print(bin(mask ^ (1 << 2)), "(bỏ phần tử 2)")
print((1 << m) - 1, "= tập đầy đủ", bin((1 << m) - 1))
print(bin(mask).count("1"), "phần tử trong tập")         # popcount

0b101 5
[True, False, True, False]
0b1 (bỏ phần tử 2)
15 = tập đầy đủ 0b1111
2 phần tử trong tập


### 3.4. Duyệt các tập con theo thứ tự tăng dần

`for mask in range(1 << m)` duyệt mọi tập con. Quan trọng: tập con `mask ^ (1 << j)` luôn **nhỏ hơn** `mask` (số nguyên), nên khi dùng DP "kéo" (pull), trạng thái nhỏ hơn đã tính xong rồi. Duyệt các bit đang bật: `rest = mask; while rest: low = rest & -rest; j = low.bit_length() - 1; rest ^= low`.

In [5]:
mask = 0b101101
rest = mask
bits = []
while rest:
    low = rest & -rest                  # bit thấp nhất đang bật
    bits.append(low.bit_length() - 1)   # chỉ số của nó
    rest ^= low
print(bin(mask), "->", bits)
print([j for j in range(6) if mask >> j & 1])     # cách ngắn hơn, cùng kết quả

0b101101 -> [0, 2, 3, 5]
[0, 2, 3, 5]


### 3.5. Mảng 2 chiều, `float("inf")` và `min`

Ta dùng `f[mask][j]`: bảng `2^m × m`. Giá trị chưa đạt được ghi `INF` (một số rất lớn); `min` luôn ưu tiên giá trị thật.

In [6]:
INF = float("inf")
m = 3
f = [[INF] * m for _ in range(1 << m)]
f[0b001][0] = 5
print(f[0b001], min(f[0b001]), min(f[0b010]))

[5, inf, inf] 5 inf


## 4. Ý tưởng nền tảng / trực giác

**Tính tay ví dụ 3.** `start = 3`; yêu cầu `A = (0, tầng 5)`, `B = (0, tầng 1)`, `C = (6, tầng 3)`.

- Thứ tự `A, B, C`: `3→5` lúc 2; `5→1` lúc 6; `1→3` lúc 8 ⇒ **8**.
- Thứ tự `B, A, C`: `3→1` lúc 2; `1→5` lúc 6; `5→3` lúc 8 ⇒ **8**.
- Thứ tự `C, A, B`: đến tầng 3 lúc 0 nhưng phải chờ tới 6; rồi `3→5` lúc 8; `5→1` lúc 12 ⇒ 12.

Có `3! = 6` thứ tự; tốt nhất là 8. Với `m = 16`, `16! ≈ 2·10^13`: không thể duyệt hết. Nhưng nhiều thứ tự dẫn tới **cùng trạng thái**: "đã phục vụ tập `S`, đang đứng ở yêu cầu cuối `j`". Ta chỉ cần giữ **thời điểm nhỏ nhất** cho trạng thái đó.

## 5. Cách 1 — Vét cạn: BFS từng giây và thử mọi hoán vị

**Cách 1a — BFS theo từng giây (nguyên văn đề).** Trạng thái `(tầng, mask đã phục vụ)`. Mỗi giây từ mỗi trạng thái đi lên/xuống/đứng yên (tầng nằm trong `[0, n-1]`), sau đó mọi yêu cầu ở đúng tầng đó và đã tới `arrival` được đánh dấu xong. Dừng ở giây đầu tiên có trạng thái mask đầy đủ. Chỉ dùng được khi `n`, `arrival` nhỏ.

**Cách 1b — Hoán vị.** Thử mọi thứ tự với `finish_time`. `O(m! · m)`.

In [7]:
from itertools import permutations

def elevator_bfs(n, start, requests):
    m = len(requests)
    full = (1 << m) - 1

    def fulfil(t, floor, mask):
        for k, (arrival, f) in enumerate(requests):
            if f == floor and t >= arrival:
                mask |= 1 << k
        return mask

    cur = {(start, fulfil(0, start, 0))}
    t = 0
    while True:
        if any(mask == full for _, mask in cur):
            return t
        t += 1
        nxt = set()
        for floor, mask in cur:
            for d in (-1, 0, 1):
                nf = floor + d
                if 0 <= nf < n:
                    nxt.add((nf, fulfil(t, nf, mask)))
        cur = nxt

def elevator_permutations(n, start, requests):
    return min(finish_time(start, order) for order in permutations(requests))

for n, start, reqs, e in EXAMPLES:
    print(elevator_bfs(n, start, reqs), elevator_permutations(n, start, reqs), e)

9 9 9
7 7 7
8 8 8


**Độ phức tạp.** BFS: `O(T · n · 2^m)` (`T` là đáp án) — chỉ cho thang nhỏ. Hoán vị: `O(m! · m)`, dùng được tới `m ≈ 8`.

**Nút thắt:** nhiều thứ tự đến cùng một trạng thái; chưa tận dụng điều đó.

## 6. Bước đột phá

**Insight 1 — thời điểm sớm hơn không bao giờ tệ hơn.** Giả sử hai kế hoạch cùng phục vụ tập `S` và cùng kết thúc ở yêu cầu `j`, kế hoạch A kết thúc ở thời điểm `tA ≤ tB`. Mọi cách đi tiếp từ B đều làm được từ A (A đứng chờ tới `tB` rồi bắt chước B), nên kết quả không tệ hơn. Vậy mỗi trạng thái chỉ cần giữ **thời điểm nhỏ nhất**.

**Insight 2 — trạng thái là `(tập đã phục vụ, yêu cầu cuối)`.** Vị trí thang bằng tầng của yêu cầu cuối. Hai thứ tự khác nhau của cùng tập `S` kết thúc ở cùng `j` gộp được vào một ô `f[S][j]`.

**Công thức.** Gọi `f[mask][j]` là thời điểm nhỏ nhất để phục vụ đúng các yêu cầu trong `mask`, yêu cầu **cuối cùng** là `j` (`j ∈ mask`). Đặt `prev = mask ^ (1 << j)`.

- Nếu `prev == 0`: `f[mask][j] = max(|start − floor_j|, arrival_j)`.
- Ngược lại: `f[mask][j] = min over j0 ∈ prev của max(f[prev][j0] + |floor_j0 − floor_j|, arrival_j)`.
- Đáp án: `min_j f[full][j]`.

**Bất biến.** `f[mask][j]` là thời điểm sớm nhất mà thang vừa phục vụ xong yêu cầu `j` sau khi đã phục vụ mọi yêu cầu còn lại của `mask`.

**Quy tắc nếu… thì…:** mỗi bước thêm một yêu cầu mới: thời điểm = `max(đến nơi, arrival)`; lấy `min` trên mọi yêu cầu trước.

## 7. Các cách tiếp theo

### 7.1. Cách 2 — DP bitmask "kéo" (bottom-up, bản nộp)

**Giải thích từng đoạn.**

1. `f` kích thước `(1 << m) × m`, khởi tạo `INF`.
2. Duyệt `mask = 1 .. 2^m − 1`. Với mỗi `j ∈ mask`: `prev = mask ^ (1 << j)`.
3. Nếu `prev == 0`, đặt giá trị khởi đầu từ `start`. Ngược lại duyệt `j0 ∈ prev` và lấy `min`.
4. Đáp án `min(f[full])`.

In [8]:
def elevator_bitmask(n, start, requests):
    m = len(requests)
    INF = float("inf")
    arr = [a for a, _ in requests]
    flo = [f for _, f in requests]
    full = (1 << m) - 1
    f = [[INF] * m for _ in range(1 << m)]
    for mask in range(1, 1 << m):
        for j in range(m):
            if not (mask >> j) & 1:
                continue
            prev = mask ^ (1 << j)
            if prev == 0:
                f[mask][j] = max(abs(start - flo[j]), arr[j])
                continue
            best = INF
            for j0 in range(m):
                if (prev >> j0) & 1:
                    cand = max(f[prev][j0] + abs(flo[j0] - flo[j]), arr[j])
                    if cand < best:
                        best = cand
            f[mask][j] = best
    return min(f[full])

for n, start, reqs, e in EXAMPLES:
    print(elevator_bitmask(n, start, reqs), e)

9 9
7 7
8 8


**Độ phức tạp.** `2^m · m` trạng thái, mỗi trạng thái duyệt `≤ m` phần tử trước ⇒ `O(m² · 2^m)` thời gian, `O(m · 2^m)` bộ nhớ. Với `m = 16`: `256 · 65536 ≈ 1.7·10^7` phép (cận trên; thực tế ít hơn vì chỉ duyệt các bit bật).

### 7.2. Cách 3 — Top-down: đệ quy có nhớ

Hỏi ngược: `solve(mask, last)` = thời điểm nhỏ nhất để **hoàn thành** mọi yêu cầu còn lại, tính từ lúc đã phục vụ `mask` và đang đứng ở `last`? Không — ở đây thời điểm là **giá trị DP**, không phải tham số, nên top-down viết theo kiểu "tập đã phục vụ + yêu cầu cuối" như bản bottom-up: `g(mask, j)` = thời điểm nhỏ nhất kết thúc ở `j` sau khi phục vụ `mask`.

In [9]:
from functools import lru_cache

def elevator_memo(n, start, requests):
    m = len(requests)
    arr = [a for a, _ in requests]
    flo = [f for _, f in requests]

    @lru_cache(maxsize=None)
    def g(mask, j):
        prev = mask ^ (1 << j)
        if prev == 0:
            return max(abs(start - flo[j]), arr[j])
        best = float("inf")
        for j0 in range(m):
            if (prev >> j0) & 1:
                best = min(best, max(g(prev, j0) + abs(flo[j0] - flo[j]), arr[j]))
        return best

    full = (1 << m) - 1
    return min(g(full, j) for j in range(m))

for n, start, reqs, e in EXAMPLES:
    print(elevator_memo(n, start, reqs), e)

9 9
7 7
8 8


**Độ phức tạp.** Cùng `O(m² · 2^m)`. Độ sâu đệ quy `≤ m = 16` nên an toàn; chỉ các trạng thái "kéo tới" từ `full` được tính (ở bài này gần như tất cả).

### 7.3. Cách 4 — DP bitmask "đẩy" chỉ duyệt các bit đang bật (nhanh hơn trong Python)

Thay vì kiểm `(mask >> j) & 1` cho mọi `j`, ta tính sẵn **ma trận khoảng cách** `dist[j][k]` và dùng "đẩy" (push): từ trạng thái `(mask, j)` thử thêm yêu cầu `k ∉ mask`. Duyệt các `k` chưa có bằng `rest = full ^ mask`. Bỏ qua trạng thái `INF`.

In [10]:
def elevator_push(n, start, requests):
    m = len(requests)
    INF = float("inf")
    arr = [a for a, _ in requests]
    flo = [f for _, f in requests]
    dist = [[abs(flo[j] - flo[k]) for k in range(m)] for j in range(m)]
    full = (1 << m) - 1
    f = [[INF] * m for _ in range(1 << m)]
    for j in range(m):
        f[1 << j][j] = max(abs(start - flo[j]), arr[j])
    for mask in range(1, 1 << m):
        row = f[mask]
        rest0 = full ^ mask
        for j in range(m):
            t = row[j]
            if t == INF:
                continue
            dj = dist[j]
            rest = rest0
            while rest:
                low = rest & -rest
                k = low.bit_length() - 1
                rest ^= low
                nt = t + dj[k]
                if nt < arr[k]:
                    nt = arr[k]
                tgt = f[mask | low]
                if nt < tgt[k]:
                    tgt[k] = nt
    return min(f[full])

for n, start, reqs, e in EXAMPLES:
    print(elevator_push(n, start, reqs), e)

9 9
7 7
8 8


**Độ phức tạp.** Vẫn `O(m² · 2^m)` về lý thuyết, nhưng hằng số nhỏ vì chỉ đi qua các cặp `(j ∈ mask, k ∉ mask)`: tổng số cặp là `Σ_mask |mask| · (m − |mask|) = m(m−1) · 2^{m−2}` — nhỏ hơn cận trên `m² · 2^m` khoảng 4 lần.

### 7.4. Cách 5 — Nhìn trước bản IV: DP đoạn `O(m²)` trên đường thẳng

Vì các tầng nằm trên một **đường thẳng**, thứ tự các **lần ghé cuối** là "bóc dần từ hai đầu" sau khi sắp theo tầng. Trạng thái `(l, r, side)` = đã bóc `l` yêu cầu từ trái và `r` từ phải, thang ở yêu cầu vừa bóc (trái hoặc phải). Chuyển: bóc trái `max(t + |pos − F[l]|, A[l])`, bóc phải `max(t + |pos − F[m−1−r]|, A[m−1−r])`. Đây là lời giải chính của bài IV (`m ≤ 500`); ở đây ta dùng để **kiểm chéo** bản bitmask.

In [11]:
def elevator_interval(n, start, requests):
    R = sorted((f, a) for a, f in requests)
    m = len(R)
    F = [x[0] for x in R]
    A = [x[1] for x in R]
    INF = float("inf")
    dp = [[[INF, INF] for _ in range(m + 1)] for _ in range(m + 1)]
    dp[1][0][0] = max(abs(start - F[0]), A[0])
    dp[0][1][1] = min(dp[0][1][1], max(abs(start - F[m - 1]), A[m - 1]))
    best = INF
    for d in range(1, m + 1):
        for l in range(d + 1):
            r = d - l
            for side in (0, 1):
                t = dp[l][r][side]
                if t == INF:
                    continue
                pos = F[l - 1] if side == 0 else F[m - r]
                if d == m:
                    best = min(best, t)
                    continue
                dp[l + 1][r][0] = min(dp[l + 1][r][0], max(t + abs(pos - F[l]), A[l]))
                dp[l][r + 1][1] = min(dp[l][r + 1][1], max(t + abs(pos - F[m - 1 - r]), A[m - 1 - r]))
    return best

for n, start, reqs, e in EXAMPLES:
    print(elevator_interval(n, start, reqs), e)

9 9
7 7
8 8


## 8. TRACE từng bước

### 8.1. Bảng `f[mask][j]` cho ví dụ 3

`start = 3`; yêu cầu `0 = (0, tầng 5)`, `1 = (0, tầng 1)`, `2 = (6, tầng 3)`. Hàm trace in từng ô theo thứ tự `mask` tăng dần, cho biết ô nào lấy từ ô nào.

| mask | j (cuối) | prev | cách tính | giá trị |
|---|---|---|---|---|
| `001` | 0 | `000` | `max(|3−5|, 0)` | 2 |
| `010` | 1 | `000` | `max(|3−1|, 0)` | 2 |
| `100` | 2 | `000` | `max(|3−3|, 6)` | 6 |
| `011` | 0 | `010` | `max(f[010][1] + |1−5|, 0)` | 6 |
| `011` | 1 | `001` | `max(f[001][0] + |5−1|, 0)` | 6 |
| `101` | 2 | `001` | `max(f[001][0] + |5−3|, 6)` | 6 |
| `111` | 2 | `011` | `min(max(6 + |5−3|, 6), max(6 + |1−3|, 6))` | **8** |

In [12]:
def trace_elevator_bitmask(n, start, requests):
    m = len(requests)
    INF = float("inf")
    arr = [a for a, _ in requests]
    flo = [f for _, f in requests]
    full = (1 << m) - 1
    f = {}
    for mask in range(1, 1 << m):
        for j in range(m):
            if not (mask >> j) & 1:
                continue
            prev = mask ^ (1 << j)
            if prev == 0:
                f[(mask, j)] = max(abs(start - flo[j]), arr[j])
                print(f"f[{mask:0{m}b}][{j}] = max(|{start}-{flo[j]}|, {arr[j]}) = {f[(mask, j)]}")
                continue
            options = []
            for j0 in range(m):
                if (prev >> j0) & 1:
                    options.append((max(f[(prev, j0)] + abs(flo[j0] - flo[j]), arr[j]), j0))
            val, who = min(options)
            f[(mask, j)] = val
            print(f"f[{mask:0{m}b}][{j}] = min over j0 in prev={prev:0{m}b}: {options} -> chọn j0={who}, giá trị {val}")
    ans = min(f[(full, j)] for j in range(m))
    print("Đáp án =", ans)
    return ans

trace_elevator_bitmask(*EXAMPLES[2][:3])

f[001][0] = max(|3-5|, 0) = 2
f[010][1] = max(|3-1|, 0) = 2
f[011][0] = min over j0 in prev=010: [(6, 1)] -> chọn j0=1, giá trị 6
f[011][1] = min over j0 in prev=001: [(6, 0)] -> chọn j0=0, giá trị 6
f[100][2] = max(|3-3|, 6) = 6
f[101][0] = min over j0 in prev=100: [(8, 2)] -> chọn j0=2, giá trị 8
f[101][2] = min over j0 in prev=001: [(6, 0)] -> chọn j0=0, giá trị 6
f[110][1] = min over j0 in prev=100: [(8, 2)] -> chọn j0=2, giá trị 8
f[110][2] = min over j0 in prev=010: [(6, 1)] -> chọn j0=1, giá trị 6
f[111][0] = min over j0 in prev=110: [(12, 1), (8, 2)] -> chọn j0=2, giá trị 8
f[111][1] = min over j0 in prev=101: [(12, 0), (8, 2)] -> chọn j0=2, giá trị 8
f[111][2] = min over j0 in prev=011: [(8, 0), (8, 1)] -> chọn j0=0, giá trị 8
Đáp án = 8


8

### 8.2. Trace ví dụ 1 (`start = 0`, `[[0,8],[6,5]]`)

Quan sát `f[11][0]` (phục vụ cả hai, cuối là yêu cầu 0 = tầng 8) = `max(f[10][1] + 3, 0)` với `f[10][1] = max(5, 6) = 6` ⇒ 9, và `f[11][1] = max(f[01][0] + 3, 6) = max(8 + 3, 6) = 11`. Chọn 9.

In [13]:
trace_elevator_bitmask(*EXAMPLES[0][:3])

f[01][0] = max(|0-8|, 0) = 8
f[10][1] = max(|0-5|, 6) = 6
f[11][0] = min over j0 in prev=10: [(9, 1)] -> chọn j0=1, giá trị 9
f[11][1] = min over j0 in prev=01: [(11, 0)] -> chọn j0=0, giá trị 11
Đáp án = 9


9

## 9. Code nộp LeetCode

Bản nộp dùng Cách 2 (DP "kéo"), bổ sung đọc các bit bật bằng danh sách để gọn và nhanh hơn một chút.

In [14]:
class Solution:
    def elevatorRequests(self, n: int, start: int, requests: list[list[int]]) -> int:
        m = len(requests)
        INF = float("inf")
        arrival = [a for a, _ in requests]
        floor = [f for _, f in requests]
        full = (1 << m) - 1

        # f[mask][j]: thời điểm nhỏ nhất đã phục vụ tập mask, yêu cầu cuối cùng là j
        f = [[INF] * m for _ in range(1 << m)]
        for mask in range(1, 1 << m):
            bits = [j for j in range(m) if (mask >> j) & 1]
            for j in bits:
                prev = mask ^ (1 << j)
                if prev == 0:
                    f[mask][j] = max(abs(start - floor[j]), arrival[j])
                    continue
                best = INF
                for j0 in bits:
                    if j0 != j:
                        cand = max(f[prev][j0] + abs(floor[j0] - floor[j]), arrival[j])
                        if cand < best:
                            best = cand
                f[mask][j] = best
        return min(f[full])


sol = Solution()
for n, start, reqs, e in EXAMPLES:
    assert sol.elevatorRequests(n, start, reqs) == e, (n, start, reqs)
print("3 ví dụ của đề: OK")

3 ví dụ của đề: OK


## 10. Vì sao thuật toán đúng

**(a) Mỗi thứ tự cho một lộ trình hợp lệ.** Công thức `max(t + khoảng_cách, arrival)` mô tả "đi thẳng tới tầng tiếp theo, chờ nếu cần". Đó là lộ trình thật nên `đáp án DP ≥ tối ưu thật`.

**(b) Tối ưu thật biểu diễn được.** Với lộ trình tối ưu bất kỳ, sắp các yêu cầu theo thời điểm **lần đầu** thang ở tầng của nó sau `arrival`. Thứ tự đó, đem chạy bằng công thức ở (a), cho thời điểm mỗi yêu cầu **không muộn hơn** thời điểm thật (quy nạp: `t_DP(k) ≤ t_thật(k)` vì lộ trình thật phải đi ít nhất khoảng cách giữa hai tầng và tôn trọng `arrival`). Vậy `đáp án DP ≤ tối ưu thật`.

**(c) Gộp trạng thái đúng.** Theo Insight 1 (mục 6), giữ thời điểm nhỏ nhất của `(mask, j)` không làm mất lời giải tối ưu.

**(d) Yêu cầu cùng tầng.** Khoảng cách giữa chúng là 0, công thức vẫn đúng (chỉ có `max` theo `arrival`).

**(e) Tương ứng với bản IV.** Cách 5 cho cùng đáp án — được kiểm chứng ở phần 12 — vì cấu trúc "bóc từ hai đầu" chỉ là một cách đếm thứ tự hợp lý trên đường thẳng.

## 11. Phân tích Time/Space

Đặt `m = len(requests)` (≤ 16).

| Cách | Time | Space |
|---|---|---|
| 1a. BFS | `O(T · n · 2^m)` | `O(n · 2^m)` |
| 1b. Hoán vị | `O(m! · m)` | `O(m)` |
| 2. DP bitmask (kéo) | `O(m² · 2^m)` | `O(m · 2^m)` |
| 3. Top-down | `O(m² · 2^m)` | `O(m · 2^m)` + ngăn xếp `m` |
| 4. DP bitmask (đẩy) | `O(m² · 2^m)`, hằng số nhỏ | `O(m · 2^m)` |
| 5. DP đoạn | `O(m²)` | `O(m²)` |

`n` và `arrival` (tới `10^9`) **không** ảnh hưởng thời gian — chỉ giá trị số. Ta đo `m = 16`.

In [15]:
import random, time
rng = random.Random(4027)
reqs16 = [[rng.randint(0, 10**9), rng.randint(0, 10**9 - 1)] for _ in range(16)]
st = rng.randint(0, 10**9 - 1)
res = {}
for name, fn in [("bitmask kéo (Solution)", Solution().elevatorRequests), ("bitmask đẩy", elevator_push),
                 ("top-down", elevator_memo), ("DP đoạn", elevator_interval)]:
    t = time.time()
    res[name] = fn(10**9, st, reqs16)
    print(f"{name:24s} m=16 -> {res[name]}  ({time.time() - t:.2f}s)")
assert len(set(res.values())) == 1

bitmask kéo (Solution)   m=16 -> 1345177363  (0.58s)


bitmask đẩy              m=16 -> 1345177363  (0.60s)


top-down                 m=16 -> 1345177363  (1.45s)
DP đoạn                  m=16 -> 1345177363  (0.00s)


## 12. Bộ test

Ví dụ đề, edge case, rồi đối chiếu các cách. **BFS theo từng giây** là trọng tài nguyên văn đề (cho thang nhỏ). Hoán vị cho `m ≤ 7`. DP đoạn kiểm chéo cho `m` lớn hơn.

In [16]:
ALL_FAST = [elevator_bitmask, elevator_memo, elevator_push, elevator_interval, Solution().elevatorRequests]

EDGE = [
    (1, 0, [[0, 0]], 0),                      # đã đứng đúng tầng
    (1, 0, [[7, 0]], 7),                      # chỉ chờ
    (5, 2, [[0, 2], [0, 2]], 0),              # hai yêu cầu cùng tầng
    (5, 0, [[0, 4]], 4),
    (5, 4, [[0, 0]], 4),
    (6, 0, [[0, 3], [10, 3]], 10),            # cùng tầng khác arrival
    (6, 3, [[0, 0], [0, 5]], 7),              # 3 -> 5 rồi 5 -> 0
    (10, 4, [[0, 0], [0, 9], [20, 4]], 20),   # bị chặn bởi arrival 20
    (10**9, 0, [[10**9, 10**9 - 1]], 10**9),  # giá trị lớn: đi 10^9 - 1, chờ tới 10^9
]
for n, s, r, e in EDGE + EXAMPLES:
    for fn in ALL_FAST:
        assert fn(n, s, r) == e, (fn.__name__, n, s, r, fn(n, s, r), e)
    if n <= 20:
        assert elevator_bfs(n, s, r) == e, (n, s, r)
print("Ví dụ đề + edge case: mọi cách đúng")

rng = random.Random(402701)
for _ in range(400):
    m = rng.randint(1, 4)
    n = rng.randint(1, 8)
    reqs = [[rng.randint(0, 8), rng.randint(0, n - 1)] for _ in range(m)]
    s = rng.randint(0, n - 1)
    ref = elevator_bfs(n, s, reqs)
    assert elevator_permutations(n, s, reqs) == ref, (n, s, reqs)
    for fn in ALL_FAST:
        assert fn(n, s, reqs) == ref, (fn.__name__, n, s, reqs, fn(n, s, reqs), ref)
print("400 input nhỏ: BFS = hoán vị = mọi DP")

Ví dụ đề + edge case: mọi cách đúng
400 input nhỏ: BFS = hoán vị = mọi DP


In [17]:
rng = random.Random(7)
for _ in range(200):
    m = rng.randint(1, 7)
    n = rng.randint(1, 50)
    reqs = [[rng.randint(0, 80), rng.randint(0, n - 1)] for _ in range(m)]
    s = rng.randint(0, n - 1)
    ref = elevator_permutations(n, s, reqs)
    for fn in ALL_FAST:
        assert fn(n, s, reqs) == ref, (fn.__name__, n, s, reqs)
print("200 input m <= 7: mọi DP khớp hoán vị")

for _ in range(40):
    m = rng.randint(8, 11)
    reqs = [[rng.randint(0, 10**9), rng.randint(0, 10**9 - 1)] for _ in range(m)]
    s = rng.randint(0, 10**9 - 1)
    a = Solution().elevatorRequests(10**9, s, reqs)
    assert a == elevator_push(10**9, s, reqs) == elevator_interval(10**9, s, reqs) == elevator_memo(10**9, s, reqs)
print("40 input m = 8..11, giá trị lớn: bitmask = đẩy = top-down = DP đoạn")

reqs = [[0, 8], [6, 5]]
snap = [r[:] for r in reqs]
Solution().elevatorRequests(9, 0, reqs)
assert reqs == snap

200 input m <= 7: mọi DP khớp hoán vị


40 input m = 8..11, giá trị lớn: bitmask = đẩy = top-down = DP đoạn


## 13. Lỗi người mới hay gặp

### Lỗi 1 — Phục vụ theo thứ tự `arrival` hoặc tham lam "tầng gần nhất"

**Vì sao sai:** thứ tự tối ưu phụ thuộc đồng thời khoảng cách và thời gian chờ. Ví dụ 1: yêu cầu xuất hiện sớm (tầng 8) nên *làm sau*.

In [18]:
def wrong_by_arrival(n, start, requests):
    return finish_time(start, sorted((a, f) for a, f in requests))

def wrong_nearest_first(n, start, requests):
    left, pos, t = list(requests), start, 0
    while left:
        left.sort(key=lambda r: (abs(pos - r[1]), r[0]))
        a, f = left.pop(0)
        t = step(t, pos, f, a); pos = f
    return t

print("theo arrival:", wrong_by_arrival(9, 0, [[0, 8], [6, 5]]), "| đúng 9")
rng = random.Random(3)
shown = 0
for _ in range(5000):
    n = rng.randint(3, 12)
    reqs = [[rng.randint(0, 12), rng.randint(0, n - 1)] for _ in range(rng.randint(2, 5))]
    s = rng.randint(0, n - 1)
    if wrong_nearest_first(n, s, reqs) != Solution().elevatorRequests(n, s, reqs):
        print("phản ví dụ cho 'gần nhất':", n, s, reqs, wrong_nearest_first(n, s, reqs), "!=", Solution().elevatorRequests(n, s, reqs))
        shown += 1
        if shown >= 2:
            break

theo arrival: 11 | đúng 9
phản ví dụ cho 'gần nhất': 3 2 [[4, 2], [3, 0], [11, 1], [8, 2], [7, 1]] 12 != 11
phản ví dụ cho 'gần nhất': 5 0 [[10, 1], [8, 3], [11, 0]] 14 != 11


### Lỗi 2 — Quên chờ: `t += khoảng_cách` thay cho `max(t + khoảng_cách, arrival)`

**Vì sao sai:** yêu cầu chưa xuất hiện thì chưa phục vụ được. **Code đúng:** `t = max(t + abs(pos - f), arrival)`.

### Lỗi 3 — Khởi tạo bằng 0 cho trạng thái đầu thay vì tính từ `start`

**Code sai:** `f[1 << j][j] = arrival[j]`. **Vì sao sai:** thang phải **đi** từ `start` tới tầng `floor[j]`. **Code đúng:** `max(abs(start - floor[j]), arrival[j])`.

### Lỗi 4 — Lẫn `mask ^ (1 << j)` khi `j` chưa thuộc `mask`

**Code sai:** dùng `^` để "bỏ `j`" khi `j` không có trong `mask` — thực chất nó **thêm** `j`.
**Code đúng:** chỉ bỏ `j` khi chắc `j ∈ mask` (hoặc dùng `mask & ~(1 << j)`).

In [19]:
mask = 0b101
print(bin(mask ^ (1 << 1)), "<- 'bỏ' phần tử 1 nhưng 1 không có: thực ra là THÊM nó")
print(bin(mask & ~(1 << 1)), "<- cách an toàn")

def wrong_no_wait(n, start, requests):
    best = float("inf")
    for order in permutations(requests):
        t, pos = 0, start
        for a, f in order:
            t += abs(pos - f)               # SAI: không chờ arrival
            pos = f
        best = min(best, t)
    return best

print("không chờ:", wrong_no_wait(9, 0, [[0, 8], [6, 5]]), "| đúng 9")

0b111 <- 'bỏ' phần tử 1 nhưng 1 không có: thực ra là THÊM nó
0b101 <- cách an toàn
không chờ: 8 | đúng 9


## 14. Mẹo & nhận dạng pattern

- **"Chọn thứ tự thăm ≤ 16–20 điểm"** → DP bitmask Held–Karp `f[tập][điểm cuối]`.
- **"Có thể chờ"** ⇒ thời điểm sớm hơn không tệ hơn ⇒ chỉ lưu `min`.
- **"Mốc thời gian chỉ có cận dưới (release time)"** → mỗi bước `max(đến nơi, mốc)`.
- Khi các điểm nằm trên **đường thẳng**, thường có DP đoạn `O(m²)` (bản IV).
- Bitmask: duyệt bit bật bằng `rest & -rest`; số trạng thái `(j ∈ mask, k ∉ mask)` là `m(m−1)·2^{m−2}`.
- Dùng BFS theo từng giây trên thang nhỏ làm **trọng tài** nguyên văn đề.

## 15. Biến thể

1. **Elevator Requests IV (m ≤ 500):** DP đoạn `O(m²)` (Cách 5).
2. **Elevator Requests II:** tất cả `arrival = 0`, cực tiểu hoá **tổng thời điểm phục vụ** (penalty) — Bài 5 làm bản bitmask.
3. **Phải về `start` sau khi xong:** cộng `abs(floor_j − start)` ở trạng thái cuối.
4. **Mỗi yêu cầu có hạn chót:** loại trạng thái có `f > deadline` (bỏ ô `INF`).
5. **Nhiều thang máy:** nhóm các yêu cầu cho từng thang (DP trên tập con), độ khó tăng.

## 16. Bảng so sánh các cách

| Cách | Ý tưởng | Time | Space | Khi nào dùng |
|---|---|---|---|---|
| 1a. BFS từng giây | Mô phỏng nguyên văn | `O(T·n·2^m)` | `O(n·2^m)` | Trọng tài, thang nhỏ |
| 1b. Hoán vị | Thử mọi thứ tự | `O(m!·m)` | `O(m)` | `m ≤ 8` |
| 2. DP bitmask kéo | `f[mask][j]` | `O(m²·2^m)` | `O(m·2^m)` | **Bản nộp** |
| 3. Top-down | Đệ quy nhớ | `O(m²·2^m)` | `O(m·2^m)` | Thích đệ quy |
| 4. DP bitmask đẩy | Chỉ duyệt bit bật | `O(m²·2^m)` (hằng nhỏ) | `O(m·2^m)` | Tăng tốc |
| 5. DP đoạn | Bóc hai đầu trên đường thẳng | `O(m²)` | `O(m²)` | `m` lớn (bản IV) |

## 17. Bài tập (5 bài, từ dễ đến khó)

**Bài 1 (đoán output).** `finish_time(2, [(0, 5), (4, 1)])` bằng bao nhiêu?

**Bài 2 (tính tay).** `n = 10`, `start = 4`, `requests = [[0,0],[0,9],[20,4]]`. Liệt kê thứ tự tốt nhất và đáp án. Vì sao yêu cầu `[20, 4]` quyết định kết quả?

**Bài 3 (điền TODO).** Hoàn thiện `bits_of(mask, m)` trả về danh sách các chỉ số bit đang bật (tăng dần).

**Bài 4 (tự viết lại).** Viết `my_elevator(n, start, requests)` theo DP bitmask.

**Bài 5 (biến thể).** **Tất cả `arrival = 0`**, cực tiểu hoá **tổng thời điểm phục vụ** (penalty, như Elevator Requests II). Viết `min_total_penalty(start, floors)` bằng DP bitmask. Gợi ý: khi đi quãng `d`, mọi yêu cầu **chưa** phục vụ (kể cả cái sắp tới) cùng chịu thêm `d`.

In [20]:
# Bài 3 — TODO (không gọi khi chạy notebook)
def bits_of(mask, m):
    # TODO: trả về [j for j in range(m) if bit j của mask bật]
    pass

# Bài 4 — TODO
def my_elevator(n, start, requests):
    # TODO: f[mask][j]; khởi đầu từ start; chuyển max(f[prev][j0] + |..|, arrival); đáp án min(f[full])
    pass

# Bài 5 — TODO
def min_total_penalty(start, floors):
    # TODO: g[mask][j] = tổng penalty nhỏ nhất; thêm yêu cầu k: cộng d * (số yêu cầu chưa phục vụ gồm cả k)
    pass

## 18. Gợi ý 3 mức

**Bài 1.**
- Mức 1: từ tầng 2 lên tầng 5 mất 3 giây.
- Mức 2: `t = max(3, 0) = 3`; xuống tầng 1 mất 4: `max(7, 4)`.
- Mức 3: `7`.

**Bài 2.**
- Mức 1: tầng 0 và tầng 9 ở hai đầu; tầng 4 ở giữa, nhưng chỉ phục vụ được từ lúc 20.
- Mức 2: `4 → 0 → 9` tốn `4 + 9 = 13`; về tầng 4 thêm 5 giây = 18. Hoặc `4 → 9 → 0` = `5 + 9 = 14`, về 4 lúc 18.
- Mức 3: về tầng 4 lúc 18 nhưng phải chờ tới 20 ⇒ đáp án **20**.

**Bài 3.**
- Mức 1: duyệt `j in range(m)`.
- Mức 2: `(mask >> j) & 1`.
- Mức 3: `[j for j in range(m) if (mask >> j) & 1]`.

**Bài 4.**
- Mức 1: xem Cách 2.
- Mức 2: `prev = mask ^ (1 << j)`; `prev == 0` thì tính từ `start`.
- Mức 3: `min` trên `j0 ∈ prev`.

**Bài 5.**
- Mức 1: nếu `r` yêu cầu chưa phục vụ thì đi quãng `d` làm tổng penalty tăng `r · d`.
- Mức 2: trạng thái `(mask, j)` với `r = m − popcount(mask)` yêu cầu còn lại *trước khi* thêm `k`.
- Mức 3: `g[mask | (1<<k)][k] = min(g[mask][j] + |fl[j] − fl[k]| · (m − popcount(mask)))` và khởi đầu `g[1<<j][j] = |start − fl[j]| · m`.

## 19. Đáp án tham khảo

In [21]:
# Bài 1, 2
assert finish_time(2, [(0, 5), (4, 1)]) == 7
assert Solution().elevatorRequests(10, 4, [[0, 0], [0, 9], [20, 4]]) == 20

# Bài 3
def bits_of_ans(mask, m):
    return [j for j in range(m) if (mask >> j) & 1]
assert bits_of_ans(0b101101, 6) == [0, 2, 3, 5]
assert bits_of_ans(0, 4) == []
print("Bài 1–3 OK")

Bài 1–3 OK


In [22]:
# Bài 4
def my_elevator_ans(n, start, requests):
    m = len(requests)
    INF = float("inf")
    f = [[INF] * m for _ in range(1 << m)]
    for mask in range(1, 1 << m):
        for j in bits_of_ans(mask, m):
            prev = mask ^ (1 << j)
            a, fl = requests[j]
            if prev == 0:
                f[mask][j] = max(abs(start - fl), a)
            else:
                f[mask][j] = min(max(f[prev][j0] + abs(requests[j0][1] - fl), a) for j0 in bits_of_ans(prev, m))
    return min(f[(1 << m) - 1])

# Bài 5
def min_total_penalty_ans(start, floors):
    m = len(floors)
    INF = float("inf")
    g = [[INF] * m for _ in range(1 << m)]
    for j in range(m):
        g[1 << j][j] = abs(start - floors[j]) * m            # m yêu cầu đều chờ quãng đầu tiên
    for mask in range(1, 1 << m):
        remaining = m - bin(mask).count("1")                 # chưa phục vụ, gồm cả yêu cầu sắp thêm
        for j in range(m):
            if g[mask][j] == INF:
                continue
            for k in range(m):
                if not (mask >> k) & 1:
                    cost = g[mask][j] + abs(floors[j] - floors[k]) * remaining
                    if cost < g[mask | (1 << k)][k]:
                        g[mask | (1 << k)][k] = cost
    return min(g[(1 << m) - 1])

def penalty_bruteforce(start, floors):
    best = float("inf")
    for order in permutations(floors):
        t, pos, tot = 0, start, 0
        for f in order:
            t += abs(pos - f); pos = f; tot += t
        best = min(best, tot)
    return best

rng = random.Random(99)
for _ in range(300):
    m = rng.randint(1, 6)
    floors = [rng.randint(0, 12) for _ in range(m)]
    s = rng.randint(0, 12)
    assert min_total_penalty_ans(s, floors) == penalty_bruteforce(s, floors), (s, floors)
    reqs = [[rng.randint(0, 9), rng.randint(0, 12)] for _ in range(rng.randint(1, 5))]
    assert my_elevator_ans(13, s, reqs) == Solution().elevatorRequests(13, s, reqs)
assert min_total_penalty_ans(4, [1, 5]) == penalty_bruteforce(4, [1, 5]) == 6   # 4 -> 5 (lúc 1), 5 -> 1 (lúc 5): 1 + 5 = 6
print("Bài 4–5 OK")

Bài 4–5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template bitmask TSP có chờ:**

```python
f[mask][j]   # thời điểm nhỏ nhất, đã phục vụ tập mask, yêu cầu cuối cùng là j
for mask in range(1, 1 << m):
    for j in bits(mask):
        prev = mask ^ (1 << j)
        if prev == 0:
            f[mask][j] = max(abs(start - floor[j]), arrival[j])
        else:
            f[mask][j] = min(max(f[prev][j0] + abs(floor[j0] - floor[j]), arrival[j]) for j0 in bits(prev))
return min(f[(1 << m) - 1])
```

**Checklist trước khi submit:**

- [ ] Thời điểm phục vụ = `max(đến nơi, arrival)`.
- [ ] Trạng thái đầu tính từ `start`.
- [ ] `prev = mask ^ (1 << j)` chỉ khi `j ∈ mask`.
- [ ] Đáp án là `min` trên yêu cầu cuối cùng của tập đầy đủ.
- [ ] Không giả sử tầng khác nhau; test hai yêu cầu cùng tầng.
- [ ] Giá trị tới `10^9`: dùng số nguyên Python, không tràn.
- [ ] Đối chiếu BFS/hoán vị trên input nhỏ có seed.